### Importing Libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

### Load Dataset

In [3]:
data1 = pd.read_csv("../DataSet/Crop_recommendation.csv")

### Encoded the Target

In [5]:
le = LabelEncoder()
data1['label'] = le.fit_transform(data1['label'])

### Spilting the Dataset into features and Targets

In [7]:
X = data1.drop('label', axis=1)
y = data1['label']

print("Shape:", X.shape)

Shape: (2200, 7)


### Train and Test Split

In [9]:
x_train, x_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

In [11]:
print("Train:", x_train.shape)
print("Test :", x_test.shape)

Train: (1760, 7)
Test : (440, 7)


### Parameter Grid

In [19]:
# These are the hyperparameters we want to search over
# RandomizedSearchCV will randomly pick combinations and test each one
param_grid = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [10, 15, 20, None],
    'min_samples_leaf':  [1, 2, 4],
    'min_samples_split': [2, 5, 10],
    'max_features':      ['sqrt', 'log2'],
    'criterion':         ['gini', 'entropy', 'log_loss']
}

### Run RandomizedSearchCV

In [22]:
# n_iter=50 means it tries 50 random combinations from the grid above
# This is faster than GridSearchCV which tries every possible combination

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf = RandomForestClassifier(random_state=42)

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=50,
    scoring='accuracy',
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(x_train, y_train)

print("\nBest Parameters:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV Accuracy: {search.best_score_:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best Parameters:
  n_estimators: 500
  min_samples_split: 5
  min_samples_leaf: 2
  max_features: log2
  max_depth: 10
  criterion: log_loss

Best CV Accuracy: 0.9955


### Evaluate Best Model on Test Set

In [34]:
best_rf = search.best_estimator_

train_acc = accuracy_score(y_train, best_rf.predict(x_train))
test_acc  = accuracy_score(y_test,  best_rf.predict(x_test))

print(f"Train Accuracy : {train_acc:.4f}")
print(f"Test  Accuracy : {test_acc:.4f}")

Train Accuracy : 1.0000
Test  Accuracy : 0.9932


### Classification Report

In [37]:
print("Classification Report:")
print(classification_report(y_test, best_rf.predict(x_test)))

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      0.95      0.97        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        20
           8       0.95      1.00      0.98        20
           9       1.00      1.00      1.00        20
          10       1.00      0.95      0.97        20
          11       0.95      1.00      0.98        20
          12       1.00      1.00      1.00        20
          13       0.95      1.00      0.98        20
          14       1.00      1.00      1.00        20
          15       1.00      1.00      1.00        20
          16       1.00      1.00      1.00        20
    

### Baseline vs Tuned Comparison

In [40]:
print("=" * 45)
print("         Model Comparison")
print("=" * 45)
print(f"  Baseline RF Test Accuracy : 0.9955")
print(f"  Tuned    RF Test Accuracy : {test_acc:.4f}")
print("=" * 45)

         Model Comparison
  Baseline RF Test Accuracy : 0.9955
  Tuned    RF Test Accuracy : 0.9932
